(cmdstanpy_conversion)=
 # Converting CmdStanPy Objects to DataTree

{class}`datatree.DataTree` is the data structure ArviZ relies on for storing inference results.

This guide demonstrates different ways to generate a `DataTree` from CmdStanPy models.

We begin by importing the required packages and defining the model using the well-known Eight Schools example.

In [4]:
import numpy as np
import arviz_base as az
from cmdstanpy import CmdStanModel

## Eight Schools Data

We use the classic Eight Schools dataset.  
`y_obs` contains the observed treatment effects for each school and  
`sigma` contains the standard errors of those estimates.

In [5]:
J = 8

# Observed treatment effects
y_obs = np.array([28.0, 8.0, -3.0, 7.0, -1.0, 1.0, 18.0, 12.0])

# Standard errors of treatment effects
sigma = np.array([15.0, 10.0, 16.0, 11.0, 9.0, 11.0, 10.0, 18.0])

In [6]:
stan_code = """
data {
  int<lower=0> J;
  array[J] real y;
  array[J] real sigma;
}

parameters {
  real mu;
  real<lower=0> tau;
  vector[J] eta;
}

transformed parameters {
  vector[J] theta = mu + tau * eta;
}

model {
  mu ~ normal(0, 5);
  tau ~ cauchy(0, 5);
  eta ~ normal(0, 1);
  y ~ normal(theta, sigma);
}

generated quantities {
  vector[J] log_lik;
  vector[J] y_rep;

  for (j in 1:J) {
    log_lik[j] = normal_lpdf(y[j] | theta[j], sigma[j]);
    y_rep[j] = normal_rng(theta[j], sigma[j]);
  }
}
"""

with open("eight_schools.stan", "w") as f:
    f.write(stan_code)

model = CmdStanModel(stan_file="eight_schools.stan")

fit = model.sample(
    data={"J": J, "y": y_obs, "sigma": sigma},
    chains=4,
    seed=123,
)

22:49:17 - cmdstanpy - INFO - compiling stan file /Users/shivanipatel/arviz-base/docs/source/how_to/eight_schools.stan to exe file /Users/shivanipatel/arviz-base/docs/source/how_to/eight_schools
22:49:24 - cmdstanpy - INFO - compiled model executable: /Users/shivanipatel/arviz-base/docs/source/how_to/eight_schools
22:49:25 - cmdstanpy - INFO - CmdStan start processing


chain 1:   0%|          | 0/2000 [00:00<?, ?it/s, (Warmup)]

chain 2:   0%|          | 0/2000 [00:00<?, ?it/s, (Warmup)]

chain 3:   0%|          | 0/2000 [00:00<?, ?it/s, (Warmup)]

chain 4:   0%|          | 0/2000 [00:00<?, ?it/s, (Warmup)]

22:49:25 - cmdstanpy - INFO - CmdStan done processing.


## Convert from CmdStanPy

This first example shows conversion from a CmdStanPy MCMC fit.

## Basic Conversion

In [7]:
idata = az.from_cmdstanpy(fit)
idata

<xarray.DataTree>
Group: /
├── Group: /posterior
│       Dimensions:        (chain: 4, draw: 1000, eta_dim_0: 8, theta_dim_0: 8,
│                           log_lik_dim_0: 8, y_rep_dim_0: 8)
│       Coordinates:
│         * chain          (chain) int64 32B 0 1 2 3
│         * draw           (draw) int64 8kB 0 1 2 3 4 5 6 ... 994 995 996 997 998 999
│         * eta_dim_0      (eta_dim_0) int64 64B 0 1 2 3 4 5 6 7
│         * theta_dim_0    (theta_dim_0) int64 64B 0 1 2 3 4 5 6 7
│         * log_lik_dim_0  (log_lik_dim_0) int64 64B 0 1 2 3 4 5 6 7
│         * y_rep_dim_0    (y_rep_dim_0) int64 64B 0 1 2 3 4 5 6 7
│       Data variables:
│           mu             (chain, draw) float64 32kB 3.706 7.042 0.8474 ... -2.2 0.9189
│           tau            (chain, draw) float64 32kB 4.344 1.882 3.843 ... 7.404 8.266
│           eta            (chain, draw, eta_dim_0) float64 256kB -0.1217 1.076 ... 1.38
│           theta          (chain, draw, theta_dim_0) float64 256kB 3.178 ... 12.33
│           log_lik        (chain, draw, log_lik_dim_0) float64 256kB -4.996 ... -3.809
│           y_rep          (chain, draw, y_rep_dim_0) float64 256kB 0.7107 ... 24.99
│       Attributes:
│           created_at:                 2026-03-15T17:19:25.194739+00:00
│           creation_library:           ArviZ
│           creation_library_version:   1.0.0
│           creation_library_language:  Python
│           inference_library:          cmdstanpy
│           inference_library_version:  1.3.0
└── Group: /sample_stats
        Dimensions:          (chain: 4, draw: 1000)
        Coordinates:
          * chain            (chain) int64 32B 0 1 2 3
          * draw             (draw) int64 8kB 0 1 2 3 4 5 6 ... 994 995 996 997 998 999
        Data variables:
            lp               (chain, draw) float64 32kB -3.693 -8.75 ... -9.715 -7.12
            acceptance_rate  (chain, draw) float64 32kB 0.9683 0.9591 ... 0.9878 1.0
            step_size        (chain, draw) float64 32kB 0.3131 0.3131 ... 0.3909 0.3909
            tree_depth       (chain, draw) int64 32kB 4 4 4 4 4 4 4 4 ... 3 3 3 2 4 4 3
            n_steps          (chain, draw) int64 32kB 15 15 15 15 15 15 ... 7 3 15 15 7
            diverging        (chain, draw) bool 4kB False False False ... False False
            energy           (chain, draw) float64 32kB 6.465 9.39 20.74 ... 14.73 13.98
        Attributes:
            created_at:                 2026-03-15T17:19:25.196578+00:00
            creation_library:           ArviZ
            creation_library_version:   1.0.0
            creation_library_language:  Python
            inference_library:          cmdstanpy
            inference_library_version:  1.3.0

## Specifying Dimensions and Coordinates

In [9]:
coords = {"school": np.arange(J)}

dims = {
    "theta": ["school"],
    "eta": ["school"],
    "log_lik": ["school"],
    "y_rep": ["school"],
}

idata_dims = az.from_cmdstanpy(
    fit,
    coords=coords,
    dims=dims,
)

idata_dims

<xarray.DataTree>
Group: /
├── Group: /posterior
│       Dimensions:  (chain: 4, draw: 1000, school: 8)
│       Coordinates:
│         * chain    (chain) int64 32B 0 1 2 3
│         * draw     (draw) int64 8kB 0 1 2 3 4 5 6 7 ... 993 994 995 996 997 998 999
│         * school   (school) int64 64B 0 1 2 3 4 5 6 7
│       Data variables:
│           mu       (chain, draw) float64 32kB 3.706 7.042 0.8474 ... 9.204 -2.2 0.9189
│           tau      (chain, draw) float64 32kB 4.344 1.882 3.843 ... 0.8872 7.404 8.266
│           eta      (chain, draw, school) float64 256kB -0.1217 1.076 ... 0.3352 1.38
│           theta    (chain, draw, school) float64 256kB 3.178 8.383 ... 3.69 12.33
│           log_lik  (chain, draw, school) float64 256kB -4.996 -3.222 ... -4.245 -3.809
│           y_rep    (chain, draw, school) float64 256kB 0.7107 4.258 ... -0.8442 24.99
│       Attributes:
│           created_at:                 2026-03-15T17:20:58.227635+00:00
│           creation_library:           ArviZ
│           creation_library_version:   1.0.0
│           creation_library_language:  Python
│           inference_library:          cmdstanpy
│           inference_library_version:  1.3.0
└── Group: /sample_stats
        Dimensions:          (chain: 4, draw: 1000)
        Coordinates:
          * chain            (chain) int64 32B 0 1 2 3
          * draw             (draw) int64 8kB 0 1 2 3 4 5 6 ... 994 995 996 997 998 999
        Data variables:
            lp               (chain, draw) float64 32kB -3.693 -8.75 ... -9.715 -7.12
            acceptance_rate  (chain, draw) float64 32kB 0.9683 0.9591 ... 0.9878 1.0
            step_size        (chain, draw) float64 32kB 0.3131 0.3131 ... 0.3909 0.3909
            tree_depth       (chain, draw) int64 32kB 4 4 4 4 4 4 4 4 ... 3 3 3 2 4 4 3
            n_steps          (chain, draw) int64 32kB 15 15 15 15 15 15 ... 7 3 15 15 7
            diverging        (chain, draw) bool 4kB False False False ... False False
            energy           (chain, draw) float64 32kB 6.465 9.39 20.74 ... 14.73 13.98
        Attributes:
            created_at:                 2026-03-15T17:20:58.232479+00:00
            creation_library:           ArviZ
            creation_library_version:   1.0.0
            creation_library_language:  Python
            inference_library:          cmdstanpy
            inference_library_version:  1.3.0

## Including Log Likelihood

In [10]:
idata_ll = az.from_cmdstanpy(
    fit,
    log_likelihood={"y": "log_lik"},
)
idata_ll

<xarray.DataTree>
Group: /
├── Group: /posterior
│       Dimensions:      (chain: 4, draw: 1000, eta_dim_0: 8, theta_dim_0: 8,
│                         y_rep_dim_0: 8)
│       Coordinates:
│         * chain        (chain) int64 32B 0 1 2 3
│         * draw         (draw) int64 8kB 0 1 2 3 4 5 6 ... 993 994 995 996 997 998 999
│         * eta_dim_0    (eta_dim_0) int64 64B 0 1 2 3 4 5 6 7
│         * theta_dim_0  (theta_dim_0) int64 64B 0 1 2 3 4 5 6 7
│         * y_rep_dim_0  (y_rep_dim_0) int64 64B 0 1 2 3 4 5 6 7
│       Data variables:
│           mu           (chain, draw) float64 32kB 3.706 7.042 0.8474 ... -2.2 0.9189
│           tau          (chain, draw) float64 32kB 4.344 1.882 3.843 ... 7.404 8.266
│           eta          (chain, draw, eta_dim_0) float64 256kB -0.1217 1.076 ... 1.38
│           theta        (chain, draw, theta_dim_0) float64 256kB 3.178 8.383 ... 12.33
│           y_rep        (chain, draw, y_rep_dim_0) float64 256kB 0.7107 4.258 ... 24.99
│       Attributes:
│           created_at:                 2026-03-15T17:22:28.818927+00:00
│           creation_library:           ArviZ
│           creation_library_version:   1.0.0
│           creation_library_language:  Python
│           inference_library:          cmdstanpy
│           inference_library_version:  1.3.0
├── Group: /sample_stats
│       Dimensions:          (chain: 4, draw: 1000)
│       Coordinates:
│         * chain            (chain) int64 32B 0 1 2 3
│         * draw             (draw) int64 8kB 0 1 2 3 4 5 6 ... 994 995 996 997 998 999
│       Data variables:
│           lp               (chain, draw) float64 32kB -3.693 -8.75 ... -9.715 -7.12
│           acceptance_rate  (chain, draw) float64 32kB 0.9683 0.9591 ... 0.9878 1.0
│           step_size        (chain, draw) float64 32kB 0.3131 0.3131 ... 0.3909 0.3909
│           tree_depth       (chain, draw) int64 32kB 4 4 4 4 4 4 4 4 ... 3 3 3 2 4 4 3
│           n_steps          (chain, draw) int64 32kB 15 15 15 15 15 15 ... 7 3 15 15 7
│           diverging        (chain, draw) bool 4kB False False False ... False False
│           energy           (chain, draw) float64 32kB 6.465 9.39 20.74 ... 14.73 13.98
│       Attributes:
│           created_at:                 2026-03-15T17:22:28.823525+00:00
│           creation_library:           ArviZ
│           creation_library_version:   1.0.0
│           creation_library_language:  Python
│           inference_library:          cmdstanpy
│           inference_library_version:  1.3.0
└── Group: /log_likelihood
        Dimensions:  (chain: 4, draw: 1000, y_dim_0: 8)
        Coordinates:
          * chain    (chain) int64 32B 0 1 2 3
          * draw     (draw) int64 8kB 0 1 2 3 4 5 6 7 ... 993 994 995 996 997 998 999
          * y_dim_0  (y_dim_0) int64 64B 0 1 2 3 4 5 6 7
        Data variables:
            y        (chain, draw, y_dim_0) float64 256kB -4.996 -3.222 ... -3.809
        Attributes:
            created_at:                 2026-03-15T17:22:28.825990+00:00
            creation_library:           ArviZ
            creation_library_version:   1.0.0
            creation_library_language:  Python
            inference_library:          cmdstanpy
            inference_library_version:  1.3.0

## Including Posterior Predictive

In [11]:
idata_pp = az.from_cmdstanpy(
    fit,
    posterior_predictive={"y": "y_rep"},
)
idata_pp

<xarray.DataTree>
Group: /
├── Group: /posterior
│       Dimensions:        (chain: 4, draw: 1000, eta_dim_0: 8, theta_dim_0: 8,
│                           log_lik_dim_0: 8)
│       Coordinates:
│         * chain          (chain) int64 32B 0 1 2 3
│         * draw           (draw) int64 8kB 0 1 2 3 4 5 6 ... 994 995 996 997 998 999
│         * eta_dim_0      (eta_dim_0) int64 64B 0 1 2 3 4 5 6 7
│         * theta_dim_0    (theta_dim_0) int64 64B 0 1 2 3 4 5 6 7
│         * log_lik_dim_0  (log_lik_dim_0) int64 64B 0 1 2 3 4 5 6 7
│       Data variables:
│           mu             (chain, draw) float64 32kB 3.706 7.042 0.8474 ... -2.2 0.9189
│           tau            (chain, draw) float64 32kB 4.344 1.882 3.843 ... 7.404 8.266
│           eta            (chain, draw, eta_dim_0) float64 256kB -0.1217 1.076 ... 1.38
│           theta          (chain, draw, theta_dim_0) float64 256kB 3.178 ... 12.33
│           log_lik        (chain, draw, log_lik_dim_0) float64 256kB -4.996 ... -3.809
│       Attributes:
│           created_at:                 2026-03-15T17:23:09.901793+00:00
│           creation_library:           ArviZ
│           creation_library_version:   1.0.0
│           creation_library_language:  Python
│           inference_library:          cmdstanpy
│           inference_library_version:  1.3.0
├── Group: /sample_stats
│       Dimensions:          (chain: 4, draw: 1000)
│       Coordinates:
│         * chain            (chain) int64 32B 0 1 2 3
│         * draw             (draw) int64 8kB 0 1 2 3 4 5 6 ... 994 995 996 997 998 999
│       Data variables:
│           lp               (chain, draw) float64 32kB -3.693 -8.75 ... -9.715 -7.12
│           acceptance_rate  (chain, draw) float64 32kB 0.9683 0.9591 ... 0.9878 1.0
│           step_size        (chain, draw) float64 32kB 0.3131 0.3131 ... 0.3909 0.3909
│           tree_depth       (chain, draw) int64 32kB 4 4 4 4 4 4 4 4 ... 3 3 3 2 4 4 3
│           n_steps          (chain, draw) int64 32kB 15 15 15 15 15 15 ... 7 3 15 15 7
│           diverging        (chain, draw) bool 4kB False False False ... False False
│           energy           (chain, draw) float64 32kB 6.465 9.39 20.74 ... 14.73 13.98
│       Attributes:
│           created_at:                 2026-03-15T17:23:09.906033+00:00
│           creation_library:           ArviZ
│           creation_library_version:   1.0.0
│           creation_library_language:  Python
│           inference_library:          cmdstanpy
│           inference_library_version:  1.3.0
└── Group: /posterior_predictive
        Dimensions:      (chain: 4, draw: 1000, y_rep_dim_0: 8)
        Coordinates:
          * chain        (chain) int64 32B 0 1 2 3
          * draw         (draw) int64 8kB 0 1 2 3 4 5 6 ... 993 994 995 996 997 998 999
          * y_rep_dim_0  (y_rep_dim_0) int64 64B 0 1 2 3 4 5 6 7
        Data variables:
            y_rep        (chain, draw, y_rep_dim_0) float64 256kB 0.7107 4.258 ... 24.99
        Attributes:
            created_at:                 2026-03-15T17:23:09.908417+00:00
            creation_library:           ArviZ
            creation_library_version:   1.0.0
            creation_library_language:  Python
            inference_library:          cmdstanpy
            inference_library_version:  1.3.0